# 02 - Bronze Auto Loader Ingestion
## 03 - Grid Events
### Goal
- copy raw files into a governed landing volume
- ingest files into a bronze Delta tables using Auto Loader

In [0]:
%run ../01_setup/00_config

In [0]:
dbutils.fs.mkdirs(grid_events_path)

for file_info in dbutils.fs.ls(f"{repo_sample_path}/grid_events"):
    if file_info.name.startswith("grid_events") and file_info.name.endswith(".csv"):
        try:
            dbutils.fs.cp(file_info.path, f"{grid_events_path}/{file_info.name}")
            print(f"Copied: {file_info.name}")
        except Exception as e:
            print(f"Copy skipped or failed: {file_info.name} - {e}")

display(dbutils.fs.ls(grid_events_path))


In [0]:
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", checkpoint_grid_events_path)
    .load(grid_events_path)
)

In [0]:
query = (
    bronze_stream_df
    .withColumn("ingestion_ts", F.current_timestamp())
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_grid_events_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_grid_events_table)
)

query.awaitTermination()
print("Bronze ingestion complete.")
display(spark.table(bronze_grid_events_table))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {bronze_grid_events_table}")
dbutils.fs.rm(checkpoint_grid_events_path, recurse=True)

In [0]:
dbutils.library.restartPython()